# Custom22 CfC Eğitimi - Google Colab

Bu notebook, 22 custom harita için CfC davranış klonlama, DAgger devam eğitimi ve eğitim sonrası değerlendirme akışını Colab üzerinde çalıştırır.

Önerilen kullanım: Colab menüsünden `Runtime > Change runtime type > GPU` seç. A100 veya H100 varsa eğitim süresi ciddi biçimde kısalır. Saf politika değerlendirmesinde `auto-waypoints` kapalı tutulur; waypoint yalnızca öğretmen veri üretiminde kullanılır.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi

## 2. Repoyu klonla ve kur

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/heimdilon/mujoco_lnn_navigation.git"
REPO_DIR = Path("/content/mujoco_lnn_navigation")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/mujoco_lnn_navigation
!git checkout master
!git pull --ff-only

In [ ]:
%env MUJOCO_GL=egl
!apt-get update -qq
!apt-get install -y -qq libegl1 libgl1
!python -m pip install -U pip
!python -m pip install -e .

İstersen kısa testleri çalıştır. Kurulumdan sonra beklenen durum: tüm testlerin geçmesi.

In [ ]:
!python -m pytest -q

## 3. Opsiyonel: Google Drive bağla

Checkpoint'i Drive'dan yüklemek veya sonuçları Drive'a kaydetmek istiyorsan `USE_DRIVE = True` yap.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path('/content/drive/MyDrive/mujoco_lnn_navigation_runs').mkdir(parents=True, exist_ok=True)

## 4. Eğitim parametreleri

`RESUME_CHECKPOINT` boşsa eğitim sıfırdan 500 BC epoch çalışır. Lokal makinedeki `step=100` checkpoint'ini Drive'a yükleyip buradan devam edeceksen `RESUME_CHECKPOINT` değerini o dosyanın Colab yolu yap ve `RESUME_EXTRA_BC_EPOCHS = 400` bırak.

In [ ]:
import shlex
import subprocess
import torch
import yaml

RUN_NAME = "cfc_radius010_custom22_dagger2"
SPLIT_CONFIG = "configs/splits/custom22_seed25462877008.yaml"
TRAIN_CONFIG = "configs/train/bc_cfc_augmented_maps.yaml"
CHECKPOINT = f"results/{RUN_NAME}/latest.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_INTERVAL = 50

# Örnek Drive checkpoint yolu:
# RESUME_CHECKPOINT = "/content/drive/MyDrive/mujoco_lnn_navigation_runs/cfc_radius010_custom22_dagger2/latest.pt"
RESUME_CHECKPOINT = ""

BC_EPOCHS_FROM_SCRATCH = 500
RESUME_EXTRA_BC_EPOCHS = 400

def run_command(cmd):
    print("$", " ".join(shlex.quote(str(part)) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 5. BC eğitimini çalıştır veya devam ettir

In [ ]:
cmd = [
    "python", "scripts/train_bc.py",
    "--split-config", SPLIT_CONFIG,
    "--train-config", TRAIN_CONFIG,
    "--run-name", RUN_NAME,
    "--save-interval", SAVE_INTERVAL,
    "--no-final-eval",
    "--device", DEVICE,
]

if RESUME_CHECKPOINT.strip():
    cmd += ["--resume", RESUME_CHECKPOINT, "--epochs", RESUME_EXTRA_BC_EPOCHS]
else:
    cmd += ["--epochs", BC_EPOCHS_FROM_SCRATCH]

run_command(cmd)

## 6. DAgger devam eğitimi

Bu hücre son BC checkpoint'inden 2 DAgger iterasyonu çalıştırır ve aynı run klasöründeki `latest.pt` dosyasını günceller.

In [ ]:
DAGGER_ITERATIONS = 2
DAGGER_ROLLOUTS_PER_MAP = 4
DAGGER_EPOCHS = 80

cmd = [
    "python", "scripts/train_bc.py",
    "--split-config", SPLIT_CONFIG,
    "--train-config", TRAIN_CONFIG,
    "--run-name", RUN_NAME,
    "--resume", CHECKPOINT,
    "--epochs", 0,
    "--dagger-iterations", DAGGER_ITERATIONS,
    "--dagger-rollouts-per-map", DAGGER_ROLLOUTS_PER_MAP,
    "--dagger-epochs", DAGGER_EPOCHS,
    "--save-interval", SAVE_INTERVAL,
    "--no-final-eval",
    "--device", DEVICE,
]

run_command(cmd)

## 7. 22 haritada saf politika değerlendirmesi

Bu değerlendirme train + holdout haritaların tamamını çalıştırır. `auto-waypoints` kullanılmaz; sonuç robot politikasının kendi davranışını gösterir.

In [ ]:
with open(SPLIT_CONFIG, "r", encoding="utf-8") as handle:
    split = yaml.safe_load(handle)

EVAL_MAPS = split["train_maps"] + split["holdout_maps"]
EVAL_RUN_NAME = f"{RUN_NAME}_eval_all22"

cmd = [
    "python", "scripts/batch_evaluate.py",
    "--map-configs", *EVAL_MAPS,
    "--checkpoint", CHECKPOINT,
    "--episodes", 4,
    "--run-name", EVAL_RUN_NAME,
    "--max-steps", 900,
    "--goal-observation-max", 10.0,
    "--device", DEVICE,
]

run_command(cmd)

## 8. Sonuç özetini oku

In [ ]:
import pandas as pd

summary_path = Path("results") / EVAL_RUN_NAME / "summary.csv"
summary = pd.read_csv(summary_path)

holdout_names = {Path(path).stem for path in split["holdout_maps"]}
summary["split"] = summary["map"].apply(lambda name: "holdout" if name in holdout_names else "train")

display(summary[["split", "map", "success_rate", "collision_rate", "timeout_rate", "mean_steps", "mean_final_distance"]])
display(summary.groupby("split")[["success_rate", "collision_rate", "timeout_rate", "mean_steps", "mean_final_distance"]].mean())

## 9. PNG/GIF görsellerini göster

In [ ]:
from IPython.display import Image, display

pngs = sorted((Path("results") / EVAL_RUN_NAME).glob("custom_map_*/rollout.png"))
gifs = sorted((Path("results") / EVAL_RUN_NAME).glob("custom_map_*/rollout.gif"))

print(f"PNG sayısı: {len(pngs)}")
for png in pngs[:6]:
    print(png)
    display(Image(filename=str(png)))

if gifs:
    print("İlk GIF:", gifs[0])
    display(Image(filename=str(gifs[0])))

## 10. Sonuçları Drive'a kaydet

Drive bağlıysa run klasörünü ve eval çıktısını `MyDrive/mujoco_lnn_navigation_runs` altına kopyalar.

In [ ]:
drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    out_dir = drive_root / "mujoco_lnn_navigation_runs"
    out_dir.mkdir(parents=True, exist_ok=True)
    !cp -r results/{RUN_NAME} {out_dir}/
    !cp -r results/{EVAL_RUN_NAME} {out_dir}/
    print("Kaydedildi:", out_dir)
else:
    print("Drive bağlı değil. USE_DRIVE=True yapıp Drive hücresini çalıştırabilirsin.")